In [6]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score

In [7]:
DATA_PATH = "D:/Final Project/Data/news.tsv"

df = pd.read_csv(DATA_PATH, sep="\t", on_bad_lines="skip")

def clean_text(text):
    text = re.sub(r'[^A-Za-z\s]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["Headline"] = df["Headline"].apply(clean_text)
df["News body"] = df["News body"].apply(clean_text)
df["Title entity"] = df["Title entity"].astype(str)

In [14]:
COUNTRIES = [
    "India","United States","China","Germany","France",
    "Brazil","Canada","Australia","Japan","UK","Russia"
]

ORG_KEYWORDS = [
    "Inc","Ltd","Corporation","Company","University",
    "Agency","Committee","Bank","Group","Technologies"
]

PERSON_PATTERN = r"^[A-Z][a-z]+(\s[A-Z][a-z]+)+$"


def infer_entity_type(text):

    if re.match(PERSON_PATTERN, text):
        return "PERSON"

    if text in COUNTRIES:
        return "LOCATION"

    if any(k in text for k in ORG_KEYWORDS):
        return "ORG"

    return "MISC"

In [15]:
def convert_to_bio(text, entity_string):

    tokens = text.split()
    tags = ["O"] * len(tokens)

    try:
        ent_dict = ast.literal_eval(entity_string)
    except:
        ent_dict = {}

    for surface, expanded in ent_dict.items():

        ent_type = infer_entity_type(expanded)

        stoks = surface.split()
        n = len(stoks)

        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == stoks:
                tags[i] = f"B-{ent_type}"
                for j in range(i+1, i+n):
                    tags[j] = f"I-{ent_type}"

    return tokens, tags


sentences = []
labels = []

for _, row in df.iterrows():
    text = row["Headline"] + " " + row["News body"]
    tokens, tags = convert_to_bio(text, row["Title entity"])
    sentences.append(tokens)
    labels.append(tags)

In [16]:
val_idx = train_test_split(
    np.arange(len(sentences)),
    test_size=0.1,
    random_state=42
)[1]

In [17]:
def rule_based_ner(text):

    entities = []

    # PERSON
    for match in re.finditer(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b", text):
        entities.append((match.group(), "PERSON"))

    # LOCATION
    for country in COUNTRIES:
        if country in text:
            entities.append((country, "LOCATION"))

    # ORG
    for match in re.finditer(r"\b[A-Z][A-Za-z]+\s(?:{})\b".format("|".join(ORG_KEYWORDS)), text):
        entities.append((match.group(), "ORG"))

    # DATE (simple year detection)
    for match in re.finditer(r"\b\d{4}\b", text):
        entities.append((match.group(), "DATE"))

    return entities

In [18]:
def convert_rule_to_bio(text, entities):

    tokens = text.split()
    tags = ["O"] * len(tokens)

    for ent_text, ent_type in entities:

        ent_tokens = ent_text.split()
        n = len(ent_tokens)

        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == ent_tokens:
                tags[i] = f"B-{ent_type}"
                for j in range(i+1, i+n):
                    tags[j] = f"I-{ent_type}"

    return tags

In [19]:
rule_preds = []
rule_true = []

for i in tqdm(val_idx, desc="Rule-based NER"):

    text = " ".join(sentences[i])

    pred_entities = rule_based_ner(text)
    pred_tags = convert_rule_to_bio(text, pred_entities)

    rule_preds.append(pred_tags)
    rule_true.append(labels[i])

Rule-based NER: 100%|██████████| 11377/11377 [00:52<00:00, 215.28it/s]


In [20]:
precision = precision_score(rule_true, rule_preds)
recall = recall_score(rule_true, rule_preds)
f1 = f1_score(rule_true, rule_preds)

print("\n RULE-BASED RESULTS ")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

# Per-entity scores
report = classification_report(rule_true, rule_preds, output_dict=True)

print("\nPer-Entity F1:")
for k, v in report.items():
    if k not in ["micro avg", "macro avg", "weighted avg"]:
        print(f"{k}: {v['f1-score']:.4f}")


 RULE-BASED RESULTS 
Precision : 0.0159
Recall    : 0.0956
F1 Score  : 0.0272

Per-Entity F1:
LOCATION: 0.1232
MISC: 0.0000
ORG: 0.0000
PERSON: 0.0278


In [21]:
CSV_PATH = "D:/Final Project/NER/NER_comparison.csv"

df_prev = pd.read_csv(CSV_PATH)

df_prev = df_prev[["Model", "Precision", "Recall", "F1"]]

new_row = {
    "Model": "Rule-Based (Multi-Entity)",
    "Precision": round(precision, 4),
    "Recall": round(recall, 4),
    "F1": round(f1, 4)
}

df_prev = pd.concat([df_prev, pd.DataFrame([new_row])], ignore_index=True)
df_prev.to_csv(CSV_PATH, index=False)

print("\n Rule-based results added!")
print(df_prev)


 Rule-based results added!
                       Model  Precision    Recall        F1
0             BiLSTM + Glove   0.857681  0.694335  0.760835
1       BiLSTM + CRF + Glove   0.889892  0.790729  0.834635
2         Transformer (BERT)   0.876800  0.900600  0.888600
3  Rule-Based (Multi-Entity)   0.015900  0.095600  0.027200


In [22]:
import pandas as pd

CSV_PATH = "D:/Final Project/NER/NER_comparison.csv"

df = pd.read_csv(CSV_PATH)

df = df.sort_values(by="F1", ascending=False).reset_index(drop=True)

df.to_csv(CSV_PATH, index=False)

print(" CSV reordered by F1 ")
print(df)

 CSV reordered by F1 
                       Model  Precision    Recall        F1
0         Transformer (BERT)   0.876800  0.900600  0.888600
1       BiLSTM + CRF + Glove   0.889892  0.790729  0.834635
2             BiLSTM + Glove   0.857681  0.694335  0.760835
3  Rule-Based (Multi-Entity)   0.015900  0.095600  0.027200
